In [ ]:
import numpy as np

In [ ]:
# gradient boosting from scratch

In [ ]:
# repeat for n_trees 
learning_rate = 0.1
y_pred = tree_stump(x, y)
residual_y = y - y_pred
y_pred_2 = tree_stump(x, residual_y)
y_pred = y_pred + learning_rate*y_pred_2

In [ ]:
## 
class Decision_Tree():
    def __init__(self, max_depth = 1, min_samples=10):
        self.min_samples = min_samples
        self.max_depth = max_depth
        self.root = None

    def gini_score_calc(self, y):
        classes = np.unique(y)
        prob_c = 0
        for c in classes:
            p_c = np.sum(y==c)
            prob_c += (p_c/len(y))**2
        gini_score = 1 - prob_c
        return gini_score 

    # assuming X is an array 
    def best_split_point (self, X, y):
        best_gini = np.inf
        
        for feature_idx in range(X.shape[1]):
            for threshold in thresholds:
                left_data_idx = X[feature_idx,:] <= threshold
                right_data_idx = X[feature_idx,:] <= threshold

                left_gini = self.gini_score_calc(y[left_data_idx])
                right_gini = self.gini_score_calc(y[right_data_idx])

                # weighted_gini
                best_gini = 
                best_feature =  
                best_threshold = 


    def build_tree(self, X, y):
        # stopping condition
        if len(y) < self.min_samples: # or depth .. whatever stopping is here
            return Node

        best_feature, best_threshold = self.best_split_point (self, X, y)
        
        # split_data
        left_mask  = 
        right_mask = 

        # traverse children
        build_tree((X[left_mask], y[left_mask]))
        build_tree((X[right_mask], y[right_mask]))

        return Node()
    
    def predict():
        pass 

    def fit(self, X, y):    
         self.root = build_tree(X,y)   




In [ ]:
n_trees = 2
learning_rate = 0.1
## Initialize X and y from the datasets 

y_pred = np.mean(y)
residual = y
for t in n_trees:
    residual = y - y_pred
    tree = Decision_Tree(max_depth = 4)
    tree.fit(X,residual)
    y_pred = y_pred + learning_rate*tree.predict(X)

# 3. At prediction time: sum all trees
# y_pred = mean(y) + lr*tree_1(x) + lr*tree_2(x) + ...

In [ ]:
## Qs for interview 
# WHY: it's the optimal 
#    constant prediction that minimizes MSE)
#    each weak learners can be deep (3-4 levesl) but not too deep -> leads to overfitting/high variance
#    small tree- underfits, no interaction between features is captured 
# What does learning rate do?	Shrinks each tree's contribution → regularization, prevents overfitting 
# How does XGBoost improve on this?	Regularization in objective, second-order gradients (Newton's method), histogram-based splitting
#   

In [ ]:
## Gardient descent for Gradient Boosting (Classification)
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class GradientBoostingClassifier:
    def __init__(self, n_trees=100, lr=0.1, max_depth=3):
        self.n_trees = n_trees
        self.lr = lr
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        # Step 1: Initialize with log-odds of mean
        p = np.mean(y)
        self.init_pred = np.log(p / (1 - p))  # log-odds

        raw_scores = np.full(len(y), self.init_pred)

        for t in range(self.n_trees):
            # Step 2: Convert raw scores to probabilities
            probs = sigmoid(raw_scores)

            # Step 3: Compute pseudo-residuals (gradient of log-loss)
            residuals = y - probs  # this is the negative gradient!

            # Step 4: Fit a REGRESSION tree to residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)

            # Step 5: Update raw scores (in log-odds space)
            raw_scores += self.lr * tree.predict(X)

            self.trees.append(tree)

    def predict_proba(self, X):
        raw_scores = np.full(X.shape[0], self.init_pred)
        for tree in self.trees:
            raw_scores += self.lr * tree.predict(X)
        return sigmoid(raw_scores)

    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs >= 0.5).astype(int)



In [ ]:
Why it works
Residuals = y - sigmoid(raw_score) — this is the negative gradient of log-loss. If true label is 1 and current probability is 0.3, residual = 0.7 (push prediction up). If probability is 0.9, residual = 0.1 (small correction).

Trees are always regression trees — even for classification. They predict continuous residuals, not classes. The classification happens only at the final sigmoid step.

Log-odds space — we accumulate predictions in log-odds, then convert to probability at the end. This is the same trick logistic regression uses.

For multiclass (k classes)
Train k trees per round (one per class), using softmax instead of sigmoid
Each class has its own running score
Residuals per class: y_one_hot[k] - softmax(scores)[k]